# Trabajo en clase — Q-Learning con FrozenLake (Actividades)

Este notebook contiene exclusivamente las Actividades 1 y 2, junto con sus dependencias.

## 1. Dependencias y funciones base (`comun.py`)

In [1]:
from IPython.core import application
import numpy as np
import gymnasium as gym
import random
import time
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

from IPython.display import HTML
from matplotlib import animation
import matplotlib.pyplot as plt

def initialize_q_table(state_space, action_space):
    return np.zeros((state_space, action_space))

def greedy_policy(Qtable, state):
    # Si hay empate entre varias acciones con el mismo Q, desempata al azar.
    max_q = np.max(Qtable[state])
    best_actions = np.flatnonzero(Qtable[state] == max_q)
    return int(np.random.choice(best_actions))

def epsilon_greedy_policy(Qtable, state, epsilon, env):
    if random.random() < epsilon:
        return env.action_space.sample()

    return greedy_policy(Qtable, state)

def play_episode(env, Q=None, random_policy=False, max_steps=100, seed=None):
    """Ejecuta un episodio y devuelve sus frames y recompensa total."""
    state, _ = env.reset(seed=seed)
    frames = [env.render()]
    total_reward = 0.0

    for _ in range(max_steps):
        if random_policy:
            action = env.action_space.sample()
        else:
            q_values = Q[state]
            max_q = np.max(q_values)

            # Desempate aleatorio entre acciones con el mismo Q.
            best_actions = np.flatnonzero(q_values == max_q)
            action = int(np.random.choice(best_actions))

        next_state, reward, terminated, truncated, _ = env.step(action)

        frames.append(env.render())
        total_reward += reward
        state = next_state

        if terminated or truncated:
            break

    return frames, total_reward

def frames_to_video(frames, interval=700):
    """Convierte una lista de frames RGB en una animación reproducible en Jupyter."""
    fig = plt.figure(figsize=(4, 4))
    plt.axis("off")

    image = plt.imshow(frames[0])

    def update(frame):
        image.set_data(frame)
        return [image]

    anim = animation.FuncAnimation(
        fig,update,
        frames=frames,
        interval=interval,
        blit=True,
        repeat=True
    )

    plt.close(fig)
    return HTML(anim.to_jshtml())

def train_q_learning(
    env,
    Qtable,
    n_episodes=5000,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
    max_steps=100,
    start_episode=0,
):
    rewards = []
    for episode in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0

        global_episode = start_episode + episode
        epsilon = min_epsilon + (
            max_epsilon - min_epsilon
        ) * np.exp(-decay_rate * global_episode)

        for _ in range(max_steps):
            action = epsilon_greedy_policy(
                Qtable, state, epsilon, env
            )

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            best_next_q = 0.0 if done else np.max(Qtable[next_state])

            td_target = reward + gamma * best_next_q
            td_error = td_target - Qtable[state, action]
            Qtable[state, action] += learning_rate * td_error

            state = next_state
            total_reward += reward
            
            if done:
                break

        rewards.append(total_reward)

    return Qtable, rewards

def evaluate_q_policy(env, Qtable, n_episodes=100, max_steps=100):
    episode_rewards = []

    for _ in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0

        for _ in range(max_steps):
            action = greedy_policy(Qtable, state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            state = next_state
            total_reward += reward

            if terminated or truncated:
                break

        episode_rewards.append(total_reward)

    return np.mean(episode_rewards), np.std(episode_rewards)

def show_frame(env, title=''):
    frame = env.render()
    plt.figure(figsize=(4, 4))
    plt.imshow(frame)
    plt.axis('off')
    plt.title(title)
    display(plt.gcf())
    plt.close()

if __name__ == "__main__":
    env = gym.make(
        "FrozenLake-v1",
        map_name="4x4",
        is_slippery=False,
        render_mode="rgb_array"
    )

    state, info = env.reset()

    print("Estado inicial:", state)
    print("Número de estados:", env.observation_space.n)
    print("Número de acciones:", env.action_space.n)


Estado inicial: 0
Número de estados: 16
Número de acciones: 4


## 2. Actividad 1 — La Q-table

In [2]:
env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=False,
    render_mode="rgb_array"
)

state, info = env.reset()

print("Estado inicial:", state)
print("Número de estados:", env.observation_space.n)
print("Número de acciones:", env.action_space.n)

state_space = env.observation_space.n
action_space = env.action_space.n

Q = initialize_q_table(state_space, action_space)

print("Q-table shape:", Q.shape)
print("Q-table inicial:")
print(Q)


Estado inicial: 0
Número de estados: 16
Número de acciones: 4
Q-table shape: (16, 4)
Q-table inicial:
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]


###  Preguntas (Actividad 1)

- **¿Cuántas filas debe tener la Q-table?**: Debe tener tantas filas como el número de estados en el entorno. En este caso (FrozenLake 4x4), son 16 filas (estados del 0 al 15).
- **¿Cuántas columnas?**: Debe tener tantas columnas como el número de acciones posibles. En FrozenLake hay 4 acciones (Left, Down, Right, Up).
- **¿Qué representa una celda $Q[s,a]$?**: Representa el valor Q o la utilidad esperada de tomar la acción 'a' estando en el estado 's', y de ahí en adelante seguir una política óptima.


## 3. Actividad 2 — Inicio del aprendizaje y Transición

In [ ]:
print("--- Ejecutando una transición ---")
state, _ = env.reset()

epsilon = 1.0
action = epsilon_greedy_policy(Q, state, epsilon, env)

next_state, reward, terminated, truncated, _ = env.step(action)

print("state      =", state)
print("action     =", action)
print("reward     =", reward)
print("next_state =", next_state)
print("done       =", terminated or truncated)


### Preguntas (Actividad 2)

- **¿Esto significa que todas las acciones son malas, o que el agente todavía no sabe nada?**: Significa que el agente todavía no sabe nada. Al iniciar en cero, el agente asume desconocimiento sobre las utilidades, no necesariamente que sean malas (a menos que todas las recompensas sean positivas).
- **¿Qué ocurre si varias acciones tienen exactamente el mismo valor máximo?**: Se debe desempatar (usualmente de forma aleatoria) entre aquellas acciones que comparten el valor máximo para evitar sesgos y permitir exploración.


### 💬 Actividad 3 — Identificar la experiencia

Escribe la experiencia anterior como:

$$
(s,a,r,s')
$$

**Experiencia:**

$$
(\quad,\quad,\quad,\quad)
$$

¿De cuál de esos cuatro elementos **no disponíamos directamente** en Value Iteration cuando hablábamos de experiencia real?


## 4. Actividad 3 - Entrenamiento y Evaluación

In [ ]:
def main():
    # 1. Crear el entorno FrozenLake
    env = gym.make(
        "FrozenLake-v1",
        map_name="4x4",
        is_slippery=False,
        render_mode="rgb_array"
    )

    state_space = env.observation_space.n
    action_space = env.action_space.n

    # 2. Inicializar la Q-table a cero
    Q = initialize_q_table(state_space, action_space)

    #  Actividad 3 - Entrenamiento y Evaluación
    # En esta actividad, ponemos a prueba el aprendizaje. En lugar de dar
    # un solo paso, ejecutamos múltiples episodios usando el algoritmo Q-learning.

    print("--- Iniciando el entrenamiento ---")
    Q, rewards = train_q_learning(
        env=env,
        Qtable=Q,
        n_episodes=5000,       # Número de iteraciones o intentos
        learning_rate=0.7,     # Tasa de aprendizaje (qué tanto reemplaza el nuevo conocimiento al viejo)
        gamma=0.95,            # Factor de descuento (importancia de las recompensas futuras)
        max_epsilon=1.0,       # Exploración inicial (100% azar)
        min_epsilon=0.05,      # Exploración mínima final (5% azar)
        decay_rate=0.001,      # Tasa a la que decae la exploración
        max_steps=100
    )
    print("Entrenamiento completado.")

    print("\nQ-table final (lo que aprendió el agente):")
    print(Q)

    # 3. Evaluar la política aprendida (ya sin explorar al azar)
    print("\n--- Evaluando la política aprendida ---")
    mean_reward, std_reward = evaluate_q_policy(env, Q, n_episodes=100, max_steps=100)
    print(f"Recompensa media en 100 episodios: {mean_reward} +/- {std_reward}")

    if mean_reward == 1.0:
        print("¡Éxito! El agente aprendió a llegar a la meta siempre (en modo no resbaladizo).")
    else:
        print("El agente aún falla en llegar a la meta. Podrían ser necesarios más episodios.")

if __name__ == "__main__":
    main()


## 4. Del azar a una política aprendida

Vamos a observar **el mismo agente en tres momentos**. Primero no sabe nada y actúa al azar; luego veremos su política después de pocas experiencias; finalmente veremos la política después del entrenamiento completo.


In [ ]:
# Guardaremos tres momentos del aprendizaje

# Momento 1: sin entrenamiento
Q_initial = initialize_q_table(
    env.observation_space.n,
    env.action_space.n
)

# Momento 2: poco entrenamiento
EARLY_EPISODES = 50
Q_early = Q_initial.copy()
Q_early, rewards_early = train_q_learning(
    env,
    Q_early,
    n_episodes=EARLY_EPISODES,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
)

# Momento 3: continuar hasta 10 000 episodios
TOTAL_EPISODES = 10000
Q_trained = Q_early.copy()
Q_trained, rewards_final = train_q_learning(
    env,
    Q_trained,
    n_episodes=TOTAL_EPISODES - EARLY_EPISODES,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
    start_episode=EARLY_EPISODES,
)

Q = Q_trained
rewards = rewards_early + rewards_final

print('Snapshots guardados:')
print('Q_initial : 0 episodios')
print(f'Q_early   : {EARLY_EPISODES} episodios')
print(f'Q_trained : {TOTAL_EPISODES} episodios')


### Momento 1 — Sin entrenamiento: random walk
Todavía no usamos la Q-table para decidir. Cada acción se selecciona aleatoriamente. Observa cómo interactúa el agente con el mundo.

In [ ]:
frames_random, reward_random = play_episode(
    env,
    Q_initial,
    random_policy=True,
    seed=7
)

print(f"Recompensa total: {reward_random}")
frames_to_video(frames_random, interval=700)


### Momento 2 — Después de pocas iteraciones

Ahora el agente usa de forma **greedy** lo que ha aprendido en `Q_early`. Todavía conoce poco del ambiente, así que su comportamiento puede ser incompleto o equivocarse.


In [ ]:
frames_early, reward_early = play_episode(
    env,
    Q_early,
    random_policy=False,
    seed=7
)

print(f"Recompensa total: {reward_early}")
frames_to_video(frames_early, interval=700)


### Momento 3 — Agente entrenado

Finalmente usamos `Q_trained`. Ya no exploramos: en cada estado el agente selecciona una de las acciones con mayor valor $Q(s,a)$.


In [ ]:
frames_trained, reward_trained = play_episode(
    env,
    Q_trained,
    random_policy=False,
    seed=7
)

print(f"Recompensa total: {reward_trained}")
frames_to_video(frames_trained, interval=700)


### 💬 Actividad — ¿Qué cambió?

Compara las tres ejecuciones. El ambiente, los estados y las acciones son los mismos. **¿Qué cambió internamente en el agente para que su comportamiento mejore?**

Observa `Q_initial`, `Q_early` y `Q_trained` y relaciona sus valores con las acciones que viste ejecutar.


### 💬 Actividad 4 — Leer una fila de Q

Selecciona un estado $s$ y observa:

$$
Q(s,0), Q(s,1), Q(s,2), Q(s,3)
$$

**Estado seleccionado:** ___1____

**Valores Q:**
Eligimos una tabla Q demo que ha tenido 5000 episodios

- Left: 0.7350918906249998
- Down: 0.0 (obstaculo o limite)
- Right: 0.8145062499999999
- Up: 0.7737809374999999

¿Cuál acción seleccionaría:

$$
\arg\max_a Q(s,a)
$$

?

Se elige la accion de Right, pues su valor es el maximo entre las 4 acciones, significando que
escogiendo esta accion, las recompensas futuras desde ese estado son mayores que las otras acciones. 

**Interpretación:**

El valor de Q(s,a) es el valor estimado de las recompensas futuras siguiendo la accion a desde el estado s manteniendo la politica. Por lo tanto tener un valor alto significa mayores recompensas futuras, con mayor posibilidad de llegar a la meta.  
>


In [ ]:

#sea s = 1
s = 1

DEMO_EPISODES = 5000
Q_demo = Q_initial.copy()
Q_demo, rewards_early = train_q_learning(
    env,
    Q_demo,
    n_episodes=DEMO_EPISODES,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
)

print("Valores Q para la tabla demo:", Q_demo[s])
print ("Left",Q_demo[1,0])
print ("Down",Q_demo[1,1])
print ("Right",Q_demo[1,2])
print ("Up",Q_demo[1,3])

#seleccionamos la accion con el valor mayor, ya que significa que las proximas acciones eligiendo esa accion 
#con la politica contribuye la mejor recompensa en futuro. 

"""Cada celda de la Tabla Q representa la estimación del valor esperado de las recompensas futuras que 
el agente obtendría si decide tomar una acción específica (action) 
estando en una situación dada (state), y a partir de ahí se comporta de acuerdo a una política determinada."""

## 5. Evaluar la política aprendida


In [ ]:
mean_reward, std_reward = evaluate_q_policy(
    env,
    Q_trained,
    n_episodes=100
)

print(f"Mean reward: {mean_reward:.3f}")
print(f"Std reward : {std_reward:.3f}")


### 💬 Actividad 5 — Exploration vs. exploitation

Durante entrenamiento usamos $\epsilon$-greedy.

Durante evaluación usamos:

$$
a=\arg\max_aQ(s,a)
$$

¿Por qué **no exploramos** durante la evaluación?

**Conclusión:**

>


## 6. Experimento: FrozenLake estocástico


In [ ]:
slippery_env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=True,
    render_mode="rgb_array"
)

Q_slippery = initialize_q_table(
    slippery_env.observation_space.n,
    slippery_env.action_space.n
)

Q_slippery, rewards_slippery = train_q_learning(
    slippery_env,
    Q_slippery,
    n_episodes=20000,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.0005,
    max_steps=100
)

mean_reward, std_reward = evaluate_q_policy(
    slippery_env,
    Q_slippery,
    n_episodes=500
)

print(f"Mean reward: {mean_reward:.3f}")
print(f"Std reward : {std_reward:.3f}")

### 💬 Actividad 6 — Determinístico vs. estocástico

Compara:

- `is_slippery=False`
- `is_slippery=True`

¿Qué cambia en el **ambiente**?

¿Qué cambia en la **ecuación de Q-Learning**?

**Discusión:**

- Ambiente:
- Algoritmo:


# Cierre de clase

Completa antes de terminar:

**1. ¿Qué almacena $Q(s,a)$?**

>

**2. ¿De dónde sale $\max_{a'}Q(s',a')$?**

> Representa la estimación del mejor valor futuro posible desde el siguiente estado $s'$. En la actualización de Q-Learning, usamos esta expresión para calcular el 'TD Target' asumiendo que, una vez en $s'$, el agente tomará la acción que maximice la recompensa esperada (política greedy), sin importar qué política esté siguiendo actualmente.

**3. ¿Por qué necesitamos $\epsilon$-greedy?**

>

**4. ¿Por qué Q-Learning es model-free?**

>Model-free es aquel que no tiene conocimiento del entorno, por lo tanto no tiene mapa de transicion. Q-Learning no contiene informacion de su entorno, empieza a recolectar informacion solo cuando realiza acciones, es decir aprende empiricamente (por prueba y error) y guarda sus aprendizajes en la tabla Q, esto lo hace model-free. 
